# F1 INSIGHT — 01 · ETL & Warehouse Exploration

Inspects the result of the ETL pipeline: how real Formula 1 data from the
**Jolpica F1 API** was extracted, transformed and idempotently loaded into the
MySQL star schema.

> Requires: MySQL running with the `f1_insight` database populated (see
> `docs/setup.md` — run `python -m etl.run --all` from `backend/` first).

In [ ]:
# Convenience setup: make the backend package importable (run from repo root).
import os, sys
ROOT = os.getcwd()
for candidate in (os.path.join(ROOT, "backend"), os.path.join(ROOT, "..", "backend")):
    if os.path.isdir(candidate):
        sys.path.insert(0, candidate)
        break
import pandas as pd
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
print("pandas", pd.__version__)


from sqlalchemy import text
from app.db import SessionLocal


def q(sql, **params):
    "Run SQL against the warehouse and return rows as dicts."
    with SessionLocal() as db:
        rows = db.execute(text(sql), params).mappings().all()
    return [dict(r) for r in rows]


def counts():
    tables = ["dim_date", "dim_driver", "dim_constructor", "dim_circuit", "dim_race",
              "fact_race_result", "fact_qualifying", "fact_lap", "fact_pit_stop",
              "fact_driver_standing", "fact_constructor_standing"]
    out = []
    total = 0
    for t in tables:
        n = q(f"SELECT COUNT(*) AS n FROM {t}")[0]["n"]
        total += n
        out.append((t, n))
    out.append(("TOTAL", total))
    return out

## Warehouse row counts & star schema

The warehouse stores **6 fact tables + 5 dimensions**:

| Dimension | Grain | Purpose |
|---|---|---|
| `dim_date` | one day | time analysis |
| `dim_driver` / `dim_constructor` / `dim_circuit` | entity | who / what / where |
| `dim_race` | one Grand Prix | conformed mini-dimension (season + round) |

| Fact | Grain | Notes |
|---|---|---|
| `fact_race_result` | driver × race | central performance fact, `position NULL` = DNF |
| `fact_qualifying` | driver × race | session times |
| `fact_lap` | driver × lap | biggest table |
| `fact_pit_stop` | driver × stop | strategy |
| `fact_driver_standing` / `fact_constructor_standing` | entity × round | championship snapshots |

All fact loads are **upserts** on natural keys, so re-running the ETL never
duplicates rows (idempotent). `fact_lap` (one row per driver per lap, seasons ≥
2015) provides most of the volume — a full load exceeds 100,000 rows.

In [ ]:
df = pd.DataFrame(counts(), columns=["table", "rows"])
df

## ETL audit — `etl_run_log`

Every pipeline run appends an audit row with extract / transform / load counts,
duplicates removed, missing values fixed and errors.

In [ ]:
df_log = pd.DataFrame(q(
    "SELECT run_id, pipeline, scope, status, records_extracted, "
    "records_transformed, records_loaded, duplicates_removed, "
    "missing_values_fixed, errors FROM etl_run_log ORDER BY run_id DESC LIMIT 10"
))
df_log

## Derived metric — `position_gain`

`position_gain = grid - position` for classified drivers: how many places each
driver gained (or lost) during the race. It feeds analytics, OLAP and mining.

In [ ]:
df_gain = pd.DataFrame(q(
    "SELECT FLOOR(season/10)*10 AS decade, "
    "       ROUND(AVG(position_gain), 3) AS avg_gain, "
    "       COUNT(*) AS n "
    "FROM fact_race_result "
    "WHERE position_gain IS NOT NULL "
    "GROUP BY FLOOR(season/10)*10 ORDER BY decade"
))
df_gain

### Visualise race volume per decade

A quick check that the historical load is complete across eras.

In [ ]:
import matplotlib.pyplot as plt

df_bydec = pd.DataFrame(q(
    "SELECT FLOOR(season/10)*10 AS decade, COUNT(DISTINCT race_id) AS races "
    "FROM dim_race GROUP BY FLOOR(season/10)*10 ORDER BY decade"
))
plt.figure(figsize=(10, 4))
plt.bar(df_bydec["decade"].astype(str), df_bydec["races"], color="#e10600")
plt.title("Races per decade in the warehouse")
plt.ylabel("races")
plt.tight_layout()
plt.show()